## Bag of Words Implementation


In [10]:
from collections import Counter
import pandas as pd
import ast

file_path = "Processed_Reviews.csv"
df = pd.read_csv(file_path, encoding="cp1252")

# choose which column to vectorize
TOKEN_COL = "lemmatized"  # or "stemmed_words"

if TOKEN_COL not in df.columns:
    raise KeyError(f"Column '{TOKEN_COL}' not found. Available columns: {df.columns.tolist()}")

def to_tokens(x):
    """
    Accepts:
      - "['a','b']"  -> ['a','b']
      - "a b c"      -> ['a','b','c']
      - "a, b, c"    -> ['a','b','c']
      - actual list  -> list
    """
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return [t for t in x if isinstance(t, str) and t.strip()]

    if not isinstance(x, str):
        return None

    s = x.strip()
    if not s:
        return None

    # try list-like string first
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return [str(t).strip() for t in val if str(t).strip()]
        except Exception:
            pass

    # fallback: split by comma if it looks comma-separated, else whitespace
    if "," in s:
        parts = [p.strip() for p in s.split(",")]
    else:
        parts = s.split()

    return [p for p in parts if p]

tokenized_reviews = df[TOKEN_COL].apply(to_tokens).dropna()

# Flatten tokens and compute frequency
all_words = [w for review in tokenized_reviews for w in review]
word_freq = Counter(all_words)
sorted_word_freq = dict(sorted(word_freq.items(), key=lambda item: item[1], reverse=True))

# Build binary vectors
vocab = list(sorted_word_freq.keys())
document_vectors = []
for review in tokenized_reviews:
    review_set = set(review)
    document_vectors.append([1 if word in review_set else 0 for word in vocab])

doc_vectors_df = pd.DataFrame(document_vectors, columns=vocab)
doc_vectors_df.to_csv("document_vectors.csv", index=False, encoding="utf-8")

word_freq_df = pd.DataFrame(sorted_word_freq.items(), columns=["Word", "Frequency"])
word_freq_df.to_csv("word_frequency.csv", index=False, encoding="utf-8")

print("Word Frequency Table (top 30):")
print(word_freq_df.head(30))
print("\nSaved: document_vectors.csv and word_frequency.csv")

Word Frequency Table (top 30):
         Word  Frequency
0     product          7
1     quality          3
2       great          2
3     amazing          2
4        love          2
5     awesome          2
6        work          2
7   perfectly          2
8        life          2
9      expect          2
10     arrive          1
11       time          1
12  packaging          1
13      amaze          1
14        buy          1
15      phone          1
16         hz          1
17    display          1
18    totally          1
19      worth          1
20        wow          1
21        bit          1
22  expensive          1
23     laptop          1
24       fine          1
25      check          1
26       full          1
27     detail          1
28   purchase          1
29      happy          1

Saved: document_vectors.csv and word_frequency.csv


## TF-IDF Implementation

In [12]:
import pandas as pd
import math
from collections import Counter
import ast

file_path = "Processed_Reviews.csv"
df = pd.read_csv(file_path, encoding="cp1252")

# Use the column you ACTUALLY have
TOKEN_COL = "lemmatized"   # or "stemmed_words"

if TOKEN_COL not in df.columns:
    raise KeyError(f"Column '{TOKEN_COL}' not found. Available columns: {df.columns.tolist()}")

def to_tokens(x):
    """
    Handles:
      - "['a','b']"  -> ['a','b']
      - "a b c"      -> ['a','b','c']
      - "a, b, c"    -> ['a','b','c']
      - list         -> list
    """
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return [t for t in x if isinstance(t, str) and t.strip()]

    if not isinstance(x, str):
        return None

    s = x.strip()
    if not s:
        return None

    # Try parsing list-like strings safely
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return [str(t).strip() for t in val if str(t).strip()]
        except Exception:
            pass

    # Fallback: split by comma or whitespace
    if "," in s:
        parts = [p.strip() for p in s.split(",")]
    else:
        parts = s.split()

    return [p for p in parts if p]

tokenized_reviews = df[TOKEN_COL].apply(to_tokens).dropna()
documents = tokenized_reviews.tolist()

def compute_tf(document):
    word_count = Counter(document)
    doc_len = len(document)
    if doc_len == 0:
        return {}
    return {word: count / doc_len for word, count in word_count.items()}

def compute_idf(documents):
    N = len(documents)
    idf = {}
    if N == 0:
        return idf

    all_words = set(word for doc in documents for word in doc)
    for word in all_words:
        df_count = sum(1 for doc in documents if word in doc)
        # df_count should never be 0 here, but guard anyway
        idf[word] = math.log(N / (df_count if df_count else 1))
    return idf

def compute_tfidf(document, idf):
    tf = compute_tf(document)
    return {word: tf_value * idf.get(word, 0.0) for word, tf_value in tf.items()}

# TF
tf_data = [compute_tf(doc) for doc in documents]
tf_df = pd.DataFrame(tf_data).fillna(0)
tf_df.to_csv("tf_scores.csv", index=False, encoding="utf-8")

# IDF
idf = compute_idf(documents)
idf_df = pd.DataFrame([idf]).fillna(0)
idf_df.to_csv("idf_scores.csv", index=False, encoding="utf-8")

# TF-IDF
tfidf_data = [compute_tfidf(doc, idf) for doc in documents]
tfidf_df = pd.DataFrame(tfidf_data).fillna(0)
tfidf_df.to_csv("tfidf_scores.csv", index=False, encoding="utf-8")

print("Saved: tf_scores.csv, idf_scores.csv, tfidf_scores.csv")
print("Docs:", len(documents), "Vocab:", len(idf))

Saved: tf_scores.csv, idf_scores.csv, tfidf_scores.csv
Docs: 13 Vocab: 54
